# OWASP Triage — fine-tuning con LoRA en Colab

Corre el pipeline completo del repo en una GPU T4 gratis.

**Antes de ejecutar nada:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.

Las celdas van en orden. La línea base (paso 2) **antes** de entrenar (paso 3), siempre.

In [ ]:
# Verificar que hay GPU. Si esto falla, cambia el tipo de entorno (ver celda anterior).
!nvidia-smi

In [ ]:
# Clonar el repo. Si aún no está en GitHub, sube la carpeta con el panel de archivos
# de la izquierda y salta esta celda.
import pathlib
REPO_URL = "https://github.com/adrianhCoder/owasp-triage-finetune.git"
if not pathlib.Path("owasp-triage-finetune").exists():
    !git clone {REPO_URL}
%cd owasp-triage-finetune

In [ ]:
%pip install -q "transformers>=4.44" "trl>=0.12" "peft>=0.13" datasets accelerate bitsandbytes google-genai

## Paso 1 — dataset sintético

Si `data/synthetic.jsonl` ya viene en el repo (generado en otra máquina), la celda lo detecta y no hace nada.

Si no existe, necesitas la API key de Gemini: panel izquierdo → icono de **llave** (Secretos) → añadir `GEMINI_API_KEY` con tu clave de https://aistudio.google.com/apikey y activar el acceso del notebook.

In [ ]:
import os, pathlib
if pathlib.Path("data/synthetic.jsonl").exists():
    n = sum(1 for _ in open("data/synthetic.jsonl"))
    print(f"data/synthetic.jsonl ya existe ({n} ejemplos), saltando generación")
else:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    !python scripts/00_generate_synthetic.py --n 300

## Paso 2 — línea base (antes de entrenar, no se salta)

Primero se fija el test set con semilla 42, luego se mide el modelo base sobre él.

In [ ]:
!python scripts/01_train_sft_lora.py --split-only
!python scripts/02_eval.py --base

## Paso 3 — entrenamiento (1–2 horas en T4)

In [ ]:
!python scripts/01_train_sft_lora.py

## Paso 4 — evaluar el modelo entrenado (mismas métricas, mismo test set)

In [ ]:
!python scripts/02_eval.py --adapter ./qwen-owasp-triage-lora

## Descargar resultados

Colab borra el disco al cerrar la sesión. Esta celda empaqueta el adaptador LoRA y los datasets generados para que no se pierdan.

In [ ]:
!zip -qr resultados.zip qwen-owasp-triage-lora data/synthetic.jsonl data/test.jsonl
from google.colab import files
files.download("resultados.zip")

Último paso manual: copiar las métricas de los pasos 2 y 4 a la tabla de la sección 5 del `README.md`.